# Synthetic Search and Auditor Verification

This notebook verifies the search-policy phases, one cumulative 40-query
trajectory, frozen checkpoints, validation-budget enforcement, and strict
separation of hidden confirmation from policy decisions.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/gfm-auditor")

if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "search-auditor",
            "https://github.com/pj-mohanty/gfm-auditor.git",
            str(REPO),
        ],
        check=True,
    )

os.chdir(REPO)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
    check=True,
)

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Repository:", REPO)
print("Python:", sys.executable)

Repository: /content/gfm-auditor
Python: /usr/bin/python3


In [2]:
from agenticls_auditor.auditor import ClosedLoopAuditor
from agenticls_auditor.mutations import Candidate
from agenticls_auditor.search.base import (
    FixedMultistagePolicy,
    RandomPolicy,
)
from agenticls_auditor.search.trajectory import run_trajectory

CHECKPOINTS = (5, 10, 20, 40)
ALLOWANCES = {5: 2, 10: 3, 20: 5, 40: 10}

In [3]:
def make_candidates(count=80):
    return [
        Candidate(
            source_id="synthetic_gene",
            sequence=f"candidate-{index:03d}",
            codon_positions=(index % 20, (index % 20) + 1),
            depth=2,
        )
        for index in range(count)
    ]


def candidate_number(candidate):
    return int(candidate.sequence.rsplit("-", 1)[1])


def discovery_score(candidate):
    index = candidate_number(candidate)
    return ((index * 37) % 101) / 100.0 - index / 1000.0


def validation_score(candidate):
    return discovery_score(candidate) * 0.95 - 0.01


def confirmation_score(candidate):
    index = candidate_number(candidate)
    return (
        discovery_score(candidate) * 0.90
        + ((index * 13) % 7) / 1000.0
    )

In [4]:
candidates = make_candidates()

policies = [
    RandomPolicy(seed=11),
    FixedMultistagePolicy(seed=11),
    ClosedLoopAuditor(seed=11),
]

runs = {
    policy.name: run_trajectory(
        policy=policy,
        candidates=candidates,
        discovery_score=discovery_score,
        validation_score=validation_score,
        confirmation_score=confirmation_score,
        max_candidates=40,
        checkpoints=CHECKPOINTS,
        validation_allowances=ALLOWANCES,
    )
    for policy in policies
}

print("Completed policies:", sorted(runs))

Completed policies: ['auditor', 'fixed_multistage', 'random']


In [5]:
for policy_name, run in runs.items():
    row = {
        "policy": policy_name,
        "queries": len(run.events),
        "unique_candidates": len(
            {event.candidate_sequence for event in run.events}
        ),
        "checkpoints": [
            snapshot.checkpoint
            for snapshot in run.checkpoints
        ],
        "validation_counts": {
            snapshot.checkpoint: snapshot.validation_actions
            for snapshot in run.checkpoints
        },
        "actions": sorted(
            {event.policy_action for event in run.events}
        ),
    }
    print(row)

{'policy': 'random', 'queries': 40, 'unique_candidates': 40, 'checkpoints': [5, 10, 20, 40], 'validation_counts': {5: 2, 10: 3, 20: 5, 40: 10}, 'actions': ['position_balanced_random']}
{'policy': 'fixed_multistage', 'queries': 40, 'unique_candidates': 40, 'checkpoints': [5, 10, 20, 40], 'validation_counts': {5: 2, 10: 3, 20: 5, 40: 10}, 'actions': ['beam', 'greedy', 'random']}
{'policy': 'auditor', 'queries': 40, 'unique_candidates': 40, 'checkpoints': [5, 10, 20, 40], 'validation_counts': {5: 2, 10: 3, 20: 5, 40: 10}, 'actions': ['beam', 'greedy', 'random', 'random_restart']}


In [6]:
for policy_name, run in runs.items():
    assert len(run.events) == 40
    assert len(
        {event.candidate_sequence for event in run.events}
    ) == 40
    assert [
        snapshot.checkpoint
        for snapshot in run.checkpoints
    ] == [5, 10, 20, 40]
    assert {
        snapshot.checkpoint: snapshot.validation_actions
        for snapshot in run.checkpoints
    } == ALLOWANCES

    confirmation_events = [
        event
        for event in run.accountant.events
        if event["event"] == "final_confirmation"
    ]
    assert len(confirmation_events) == 4

    if policy_name == "auditor":
        assert all(
            "confirmation_margin" not in decision
            for decision in policies[2].state.decisions
        )

print("All trajectory and separation checks passed.")

All trajectory and separation checks passed.


In [7]:
subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)